# 🏋️ LoRA 학습 데모

이 노트북에서는 미리 준비된 소크라테스식 튜터링 예시 데이터셋으로 LoRA(Low-Rank Adaptation) 학습을 시연합니다.

> ⚠️ **워크숍 범위:** 미리 준비된 학습 데이터로 아주 작은 모델(`Qwen2-0.5B`)을 CPU에서 학습합니다. 이 데모는 *과정*을 보여 주기 위한 것이며, 프로덕션 학습에서는 더 큰 모델을 GPU에서 더 많은 데이터로 학습합니다.

**학습 데이터:** `socratic-training-data-curated.jsonl`을 사용합니다. 튜터가 답을 직접 주는 대신 생각을 이끄는 질문으로 응답하는, 엄선된 고품질 소크라테스식 튜터링 예시 30개로 구성된 데이터셋입니다.

다음을 진행합니다.
1. 미리 준비된 학습 데이터 불러오기
2. LoRA 파라미터 설정하기
3. LoRA 어댑터 **실제로 학습하기**(CPU에서 약 5~10분 소요)
4. 학습된 어댑터 저장하기

---
## 0. 설정

model-optimization 모듈에서 사용한 것과 같은 작은 모델을 사용합니다.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# model-optimization 모듈에서 사용한 것과 같은 작은 모델
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "./socratic-tutor-lora"
TRAINING_DATA = "socratic-training-data-curated.jsonl"  # 미리 준비된 엄선 데이터셋

print(f"Base model: {MODEL_ID}")
print(f"Training data: {TRAINING_DATA}")
print(f"Output directory: {OUTPUT_DIR}")

---
## 1. 학습 데이터 불러오기

미리 준비된 고품질 소크라테스식 튜터링 예시 30개로 구성된 데이터셋을 사용합니다. 각 예시는 다음과 같이 구성됩니다.
- 학생이 수학 질문을 합니다
- 튜터가 **생각을 이끄는 질문**으로 응답합니다(답을 직접 주지 않음)

프로덕션에서 직접 작성하거나 LLM이 생성한 데이터를 품질 기준으로 필터링해 만드는 것이 바로 이런 엄선된 데이터셋입니다.

In [ ]:
import json

# 미리 준비된 학습 데이터 불러오기
training_examples = []
with open(TRAINING_DATA, "r") as f:
    for line in f:
        if line.strip():  # 빈 줄 건너뛰기
            training_examples.append(json.loads(line))

print(f"✅ Loaded {len(training_examples)} training examples from {TRAINING_DATA}")
print(f"\nExample conversations:")
for i, ex in enumerate(training_examples[:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"  Student: {ex['messages'][0]['content']}")
    print(f"  Tutor: {ex['messages'][1]['content']}")

In [ ]:
# 학습용으로 데이터 형식 변환(messages 형식을 텍스트로 변환)
from datasets import Dataset

# 소크라테스식 동작을 정의하는 시스템 프롬프트 - 모든 예시에 포함됨
SYSTEM_PROMPT = """You are a Socratic math tutor. Never give direct answers. Instead:
- Ask guiding questions to help students discover the solution
- Encourage critical thinking and problem-solving
- Be supportive and patient"""

def format_for_training(example):
    """예시를 시스템 프롬프트가 포함된 Qwen용 채팅 대화 형식으로 변환합니다."""
    user_msg = example['messages'][0]['content']
    assistant_msg = example['messages'][1]['content']
    # 동작을 강화하기 위해 학습에 시스템 프롬프트 포함
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    }

# 데이터 증강으로 데이터셋 생성(예시를 반복해 학습 신호 강화)
formatted_data = []
AUGMENTATION_FACTOR = 3  # 각 예시를 3번 반복

for example in training_examples:
    for _ in range(AUGMENTATION_FACTOR):
        formatted_data.append(format_for_training(example))

train_dataset = Dataset.from_list(formatted_data)

print(f"Original examples: {len(training_examples)}")
print(f"After {AUGMENTATION_FACTOR}x augmentation: {len(train_dataset)} training samples")
print(f"\nFormatted example:")
print(train_dataset[0]['text'])

---
## 2. 기본 모델 불러오기

작은 Qwen 모델을 불러옵니다. model-optimization 모듈에서 사용한 것과 같은 모델입니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading base model (this may take a minute)...")

# 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# CPU에 모델 불러오기
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,  # CPU에서는 float32 사용
    device_map="cpu"
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"   Parameters: {model.num_parameters():,}")

---
## 3. LoRA 설정하기

LoRA 어댑터 설정을 구성합니다.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoRA 설정 - 빠른 학습을 위해 작은 rank 사용
lora_config = LoraConfig(
    r=8,                      # Rank - 높을수록 표현력과 메모리 사용량이 커짐, CPU 데모용으로 작게 설정
    lora_alpha=16,            # 스케일링 계수(보통 2×r)
    lora_dropout=0.1,         # 일반화 성능을 높이기 위한 정규화
    target_modules=[          # 어댑터를 붙일 레이어
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")

In [ ]:
# 모델에 LoRA 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 표시
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

---
## 4. 모델 학습하기

이제 엄선된 데이터로 실제 학습을 해 보겠습니다! CPU에서 **5~10분** 정도 걸립니다.

> ☕ 커피 한잔하기 좋은 타이밍입니다!

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 데이터셋 토큰화
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # masked LM이 아닌 Causal LM
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")

In [ ]:
# 학습 인자 - 작은 데이터셋에서 효과적으로 학습하도록 최적화
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,              # 과정을 보여 주기 위해 1 epoch만 학습
    per_device_train_batch_size=2,   # 조금 더 큰 배치
    gradient_accumulation_steps=4,   # gradient 누적
    learning_rate=2e-4,              # 빠른 학습을 위해 조금 높은 학습률
    warmup_ratio=0.1,                # 10% warmup
    logging_steps=10,
    save_strategy="no",
    fp16=False,                      # CPU에서는 FP16 사용 안 함
    optim="adamw_torch",
    report_to="none",                # wandb 등 비활성화
    lr_scheduler_type="cosine",      # Cosine 스케줄은 파인튜닝에 잘 맞음
)

print("Training Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LR scheduler: {training_args.lr_scheduler_type}")

In [ ]:
# trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Training on {len(train_dataset)} samples...")
print("This will take approximately 5-10 minutes on CPU.\n")

# 학습!
trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")

---
## 5. 어댑터 저장하기

평가와 배포에 사용할 수 있도록 학습된 LoRA 어댑터를 저장합니다.

In [ ]:
# 어댑터 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Adapter saved to: {OUTPUT_DIR}")

# 저장된 내용 표시
print("\nSaved files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    if size > 1024*1024:
        print(f"  {f}: {size/1024/1024:.1f} MB")
    elif size > 1024:
        print(f"  {f}: {size/1024:.1f} KB")
    else:
        print(f"  {f}: {size} B")

---
## 요약

실제 LoRA 파인튜닝을 완료했습니다! 지금까지 한 일은 다음과 같습니다.

1. ✅ **학습 데이터 불러오기** — 엄선된 소크라테스식 튜터링 예시 30개
2. ✅ **LoRA 설정하기** — rank, alpha, target module 설정
3. ✅ **어댑터 학습하기** — CPU에서 실제 학습 실행
4. ✅ **어댑터 저장하기** — 평가 준비 완료

실습 안내 문서로 돌아가 새로 튜닝한 모델을 평가하는 방법을 알아보세요.